# 05 — Dataset Imputation — `bridge_ml_dataset_neu`

This notebook is the single imputation step between Notebook 04 and Notebook 06. It loads the canonical NEU bridge dataset from Notebook 04, applies the defined imputation policy once, creates the canonical imputed dataset, and produces audit/provenance artifacts.

**Imputation policy**
- Numeric columns → Median
- Categorical/text columns → Mode
- Identifier and target columns → Not imputed
- The original NEU CSV is never overwritten

**Outputs**
- `bridge_ml_dataset_imputed.csv` — canonical downstream dataset
- `bridge_ml_dataset_imputed.parquet` — analytical copy
- `bridge_ml_dataset_imputation_report.xlsx` — audit report
- `bridge_ml_dataset_imputed_data_inventory.csv` — column-level inventory
- `05_data_manifest.txt` — source/transfer manifest

Notebook 06 consumes the imputed output and does not perform imputation again.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


## 1. Input and output paths

In [2]:
# 05 — Portable input/output configuration
import os

def find_project_root():
    env_root = os.getenv("BRIDGE_PROJECT_ROOT")
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / "Dataset_PlanA-B").exists():
            return root
        raise FileNotFoundError(
            f"BRIDGE_PROJECT_ROOT does not contain Dataset_PlanA-B: {root}"
        )

    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "Dataset_PlanA-B").exists():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Set BRIDGE_PROJECT_ROOT to the project folder."
    )

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "Dataset_PlanA-B"
OUTPUT_ROOT = PROJECT_ROOT / "Output_PlanA-B"

# Notebook 04 is the canonical upstream producer.
INPUT_DIR = OUTPUT_ROOT / "04_Integrated_Dataset"

# Notebook 05 owns its own outputs.
OUTPUT_DIR = OUTPUT_ROOT / "05_Dataset_Imputation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = INPUT_DIR / "bridge_ml_dataset_neu.csv"
OUTPUT_PATH = OUTPUT_DIR / "bridge_ml_dataset_imputed.csv"
OUTPUT_PARQUET_PATH = OUTPUT_DIR / "bridge_ml_dataset_imputed.parquet"
REPORT_PATH = OUTPUT_DIR / "bridge_ml_dataset_imputation_report.xlsx"
DATA_INVENTORY_PATH = OUTPUT_DIR / "bridge_ml_dataset_imputed_data_inventory.csv"
MANIFEST_PATH = OUTPUT_DIR / "05_data_manifest.txt"

print("Project root:", PROJECT_ROOT)
print("Input :", INPUT_PATH)
print("Output:", OUTPUT_PATH)
print("Parquet:", OUTPUT_PARQUET_PATH)
print("Report:", REPORT_PATH)


Project root: C:\Datenanalyse\final Project
Input : C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.csv
Output: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.csv
Parquet: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.parquet
Report: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputation_report.xlsx


## 1A. DATA SOURCE / INPUT–OUTPUT MANIFEST

| Layer | Source / origin | Transfer method | Role in Notebook 05 | Destination |
|---|---|---|---|---|
| NEU dataset | Notebook 04 `bridge_ml_dataset_neu.csv` | Local file read | Canonical pre-imputation input | `INPUT_PATH` |
| Imputation | Notebook 05 dataframe | pandas | Numeric → median; categorical/text → mode; protected columns unchanged | In-memory `df` |
| Imputed CSV | Notebook 05 `df` | Local file write | Canonical downstream dataset | `Output_PlanA-B/05_Dataset_Imputation/bridge_ml_dataset_imputed.csv` |
| Imputed Parquet | Notebook 05 `df` | Local file write | Analytical/cache copy | `Output_PlanA-B/05_Dataset_Imputation/bridge_ml_dataset_imputed.parquet` |
| Imputation report | Before/after + methods | Excel via openpyxl | Audit/provenance | `bridge_ml_dataset_imputation_report.xlsx` |
| Data inventory | Final imputed dataframe | Generated from actual dataframe | Column-level audit | `bridge_ml_dataset_imputed_data_inventory.csv` |
| Manifest | Notebook 05 configuration | Generated text file | Source/transfer documentation | `05_data_manifest.txt` |

### Transfer chain
```text
Notebook 04
    ↓
bridge_ml_dataset_neu.csv
    ↓
Notebook 05
    ├── imputation
    ├── audit
    └── data inventory
    ↓
bridge_ml_dataset_imputed.csv / .parquet
    ↓
Notebook 06
```

**Important:** Notebook 05 does not download BASt, DWD, or Traffic data and does not read PostgreSQL directly. It consumes the canonical integrated dataset produced by Notebook 04.

**Important:** The original NEU dataset is never overwritten.


## 2. Load the NEU dataset

In [3]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Input dataset not found: {INPUT_PATH}")

df = pd.read_csv(INPUT_PATH, low_memory=False)

print("Dataset loaded successfully")
print("Rows   :", len(df))
print("Columns:", len(df.columns))
display(df.head())


Dataset loaded successfully
Rows   : 52214
Columns: 97


,bridge_id,bauwerk,bauwerksart_text,stadium_text,bwnr,tbwnr,jast_lage,baujahr,laenge,breite,...,traffic_dtv_min,traffic_dtv_std,traffic_heavy_vehicle_mean,traffic_heavy_vehicle_max,traffic_heavy_vehicle_share_mean,traffic_dtv_yoy_growth_mean,traffic_dtv_yoy_growth_max,traffic_dtv_yoy_growth_min,traffic_dtv_trend_per_year,traffic_traffic_data_coverage
0,1019500._0,Brücke ...,Plattenbrücke ...,Bauwerk unter Verkehr ...,1019500,0,O: Bundesstraße,1973,4.50,14.00,...,4204.0,545.046905,223.238095,399.0,0.040557,0.01564,0.323739,-0.254949,22.471824,0.954545
1,1119503._0,Brücke ...,Brücke als geschlossener Rahmen ...,Bauwerk unter Verkehr ...,1119503,0,O: Bundesstraße,1961,3.47,14.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1119505._0,Brücke ...,Plattenbrücke ...,Bauwerk unter Verkehr ...,1119505,0,O: Bundesstraße,1962,4.50,14.00,...,4204.0,545.046905,223.238095,399.0,0.040557,0.01564,0.323739,-0.254949,22.471824,0.954545
3,1119512._0,Brücke ...,Brücke als geschlossener Rahmen ...,Bauwerk unter Verkehr ...,1119512,0,O: Bundesstraße,1981,3.60,22.53,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1119514._0,Brücke ...,Plattenbrücke ...,Bauwerk unter Verkehr ...,1119514,0,E: Bundesstraße,1986,11.04,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Columns that must NOT be imputed

These columns retain their original missing values because filling them would create artificial identifiers or target values.

In [4]:
PROTECTED_COLUMNS = {
    "bridge_id",
    "bwnr",
    "tbwnr",
    "zustandsnote",
    "zustandsnotenklasse",
    "zn"
}

existing_protected = sorted(PROTECTED_COLUMNS.intersection(df.columns))
print("Protected columns present:")
print(existing_protected)


Protected columns present:
['bridge_id', 'bwnr', 'tbwnr', 'zustandsnote', 'zustandsnotenklasse']


## 4. Missingness before imputation

In [5]:
missing_before = df.isna().sum()

before_table = pd.DataFrame({
    "column": df.columns,
    "missing_before": [int(missing_before[c]) for c in df.columns],
    "missing_pct_before": [100 * missing_before[c] / len(df) for c in df.columns]
}).sort_values("missing_before", ascending=False)

print("Total missing cells BEFORE:", int(missing_before.sum()))
display(before_table.head(30))


Total missing cells BEFORE: 556979


,column,missing_before,missing_pct_before
92,traffic_dtv_yoy_growth_mean,21372,40.931551
93,traffic_dtv_yoy_growth_max,21372,40.931551
94,traffic_dtv_yoy_growth_min,21372,40.931551
88,traffic_dtv_std,21290,40.774505
95,traffic_dtv_trend_per_year,21290,40.774505
81,traffic_traffic_last_year,21058,40.330180
91,traffic_heavy_vehicle_share_mean,21058,40.330180
84,traffic_dtv_latest,21058,40.330180
89,traffic_heavy_vehicle_mean,21058,40.330180
86,traffic_dtv_max,21058,40.330180


## 5. Imputation

**Numeric → Median**  
**Categorical/text → Mode**  
**Protected columns → Not imputed**

In [6]:
imputation_log = []

for column in df.columns:
    missing_count = int(df[column].isna().sum())

    if missing_count == 0:
        continue

    value_used = None

    # Protected columns: do not fill
    if column in PROTECTED_COLUMNS:
        method = "NOT_IMPUTED_PROTECTED"

    # Numeric columns: median
    elif pd.api.types.is_numeric_dtype(df[column]):
        value = df[column].median()

        if pd.notna(value):
            df[column] = df[column].fillna(value)
            value_used = value
            method = "MEDIAN"
        else:
            method = "NOT_IMPUTED_NO_VALID_MEDIAN"

    # Categorical/text columns: mode
    else:
        mode_values = df[column].mode(dropna=True)

        if len(mode_values) > 0:
            value = mode_values.iloc[0]
            df[column] = df[column].fillna(value)
            value_used = value
            method = "MODE"
        else:
            method = "NOT_IMPUTED_NO_VALID_MODE"

    imputation_log.append({
        "column": column,
        "missing_before": missing_count,
        "method": method,
        "value_used": value_used,
        "missing_after": int(df[column].isna().sum())
    })

imputation_report = pd.DataFrame(imputation_log)
display(imputation_report)


,column,missing_before,method,value_used,missing_after
0,breite,1,MEDIAN,14.44,0
1,trag_l_idx,4,MODE,II,0
2,gis_station_m,852,MEDIAN,1239.0,0
3,gis_road_context,232,MODE,A 3,0
4,gis_ort,247,MODE,Dortmund ...,0
...,...,...,...,...,...
77,traffic_dtv_yoy_growth_mean,21372,MEDIAN,0.051943,0
78,traffic_dtv_yoy_growth_max,21372,MEDIAN,0.769918,0
79,traffic_dtv_yoy_growth_min,21372,MEDIAN,-0.437084,0
80,traffic_dtv_trend_per_year,21290,MEDIAN,124.026523,0


## 6. Verify missingness after imputation

In [7]:
missing_after = df.isna().sum()

comparison = pd.DataFrame({
    "column": df.columns,
    "missing_before": [int(missing_before[c]) for c in df.columns],
    "missing_after": [int(missing_after[c]) for c in df.columns]
})

comparison["filled_cells"] = (
    comparison["missing_before"] - comparison["missing_after"]
)

comparison["status"] = np.select(
    [comparison["missing_after"] > 0, comparison["filled_cells"] > 0],
    ["REMAINING_NULL", "IMPUTED"],
    default="COMPLETE"
)
comparison = comparison.sort_values("filled_cells", ascending=False)

print("Total missing cells BEFORE:", int(missing_before.sum()))
print("Total missing cells AFTER :", int(missing_after.sum()))
print("Cells filled              :", int(missing_before.sum() - missing_after.sum()))

display(comparison.head(30))


Total missing cells BEFORE: 556979
Total missing cells AFTER : 0
Cells filled              : 556979


,column,missing_before,missing_after,filled_cells,status
92,traffic_dtv_yoy_growth_mean,21372,0,21372,IMPUTED
93,traffic_dtv_yoy_growth_max,21372,0,21372,IMPUTED
94,traffic_dtv_yoy_growth_min,21372,0,21372,IMPUTED
88,traffic_dtv_std,21290,0,21290,IMPUTED
95,traffic_dtv_trend_per_year,21290,0,21290,IMPUTED
81,traffic_traffic_last_year,21058,0,21058,IMPUTED
91,traffic_heavy_vehicle_share_mean,21058,0,21058,IMPUTED
84,traffic_dtv_latest,21058,0,21058,IMPUTED
89,traffic_heavy_vehicle_mean,21058,0,21058,IMPUTED
86,traffic_dtv_max,21058,0,21058,IMPUTED


## 7. Final quality checks

In [8]:
# Final structural and policy checks
original_columns = pd.read_csv(INPUT_PATH, nrows=0).columns.tolist()

assert df.columns.tolist() == original_columns, "Column structure changed during imputation."
assert len(df) == len(pd.read_csv(INPUT_PATH, usecols=["bridge_id"])) if "bridge_id" in df.columns else True

if "bridge_id" in df.columns:
    duplicate_bridge_ids = int(df["bridge_id"].duplicated().sum())
    print("Duplicate bridge_id:", duplicate_bridge_ids)
    assert duplicate_bridge_ids == 0

for protected in PROTECTED_COLUMNS.intersection(df.columns):
    before_nulls = int(missing_before[protected])
    after_nulls = int(df[protected].isna().sum())
    assert before_nulls == after_nulls, f"Protected column changed: {protected}"

print("Remaining NULL cells:", int(df.isna().sum().sum()))
print("Final shape:", df.shape)
print("Structural and protection checks: PASS")


Duplicate bridge_id: 0
Remaining NULL cells: 0
Final shape: (52214, 97)
Structural and protection checks: PASS


## 8. Save the new dataset

The original `bridge_ml_dataset_neu.csv` is not overwritten.

In [9]:
# Save canonical imputed dataset and audit artifacts

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

df.to_parquet(
    OUTPUT_PARQUET_PATH,
    index=False
)

with pd.ExcelWriter(REPORT_PATH, engine="openpyxl") as writer:
    comparison.to_excel(writer, index=False, sheet_name="Before_After")
    imputation_report.to_excel(writer, index=False, sheet_name="Imputation_Methods")

# Column-level data inventory
data_inventory = pd.DataFrame({
    "column_order": range(1, len(df.columns) + 1),
    "column": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_before": [int(missing_before[c]) for c in df.columns],
    "missing_after": [int(df[c].isna().sum()) for c in df.columns],
    "imputation_method": [
        (
            imputation_report.loc[imputation_report["column"] == c, "method"].iloc[0]
            if c in set(imputation_report["column"])
            else "NO_MISSING_VALUES"
        )
        for c in df.columns
    ],
    "protected": [c in PROTECTED_COLUMNS for c in df.columns],
})
data_inventory.to_csv(DATA_INVENTORY_PATH, index=False)

# Human-readable manifest
manifest_text = f"""Notebook 05 — Dataset Imputation Data Manifest
=================================================

PROJECT_ROOT: {PROJECT_ROOT}
DATASET_ROOT: {DATASET_ROOT}
INPUT_DIR:    {INPUT_DIR}
OUTPUT_DIR:   {OUTPUT_DIR}

INPUT
-----
Notebook 04 canonical output:
{INPUT_PATH}

PROCESSING
----------
Numeric columns: median imputation.
Categorical/text columns: mode imputation.
Protected columns: not imputed.
Original NEU dataset is never overwritten.
No PostgreSQL loading or ML training is performed here.

PROTECTED COLUMNS
-----------------
{sorted(PROTECTED_COLUMNS)}

OUTPUTS
-------
CSV:       {OUTPUT_PATH}
Parquet:   {OUTPUT_PARQUET_PATH}
Report:    {REPORT_PATH}
Inventory: {DATA_INVENTORY_PATH}
Manifest:  {MANIFEST_PATH}

DOWNSTREAM
----------
Notebook 06 consumes the imputed dataset.
"""

MANIFEST_PATH.write_text(manifest_text, encoding="utf-8")

print("========================================")
print("IMPUTATION COMPLETE")
print("========================================")
print("Input dataset :", INPUT_PATH)
print("New CSV       :", OUTPUT_PATH)
print("New Parquet   :", OUTPUT_PARQUET_PATH)
print("Audit report  :", REPORT_PATH)
print("Data inventory:", DATA_INVENTORY_PATH)
print("Data manifest :", MANIFEST_PATH)


IMPUTATION COMPLETE
Input dataset : C:\Datenanalyse\final Project\Output_PlanA-B\04_Integrated_Dataset\bridge_ml_dataset_neu.csv
New CSV       : C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.csv
New Parquet   : C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed.parquet
Audit report  : C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputation_report.xlsx
Data inventory: C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\bridge_ml_dataset_imputed_data_inventory.csv
Data manifest : C:\Datenanalyse\final Project\Output_PlanA-B\05_Dataset_Imputation\05_data_manifest.txt
